In [12]:
import pandas as pd
import sys
import torch
from termcolor import colored
from smiles_embedding import build_vocab_util
sys.modules["build_vocab_util"] = build_vocab_util
from smiles_embedding.build_vocab_util import WordVocab
from MCPdataset import load_ind_data
from Mymodel import DrugTargetModel
from until import evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def load_model(new_model, path_pretrain_model):
    pretrained_dict = torch.load(path_pretrain_model, map_location=torch.device('cpu'))
    new_model_dict = new_model.state_dict()
    pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in new_model_dict}
    new_model_dict.update(pretrained_dict)
    new_model.load_state_dict(new_model_dict)
    return new_model

In [13]:
if __name__ == '__main__':
    file = 'example_data/example_data.csv'
    vocab = WordVocab.load_vocab('example_data/example_data_vocab.pkl')
    test_iter = load_ind_data(file, vocab)
    original_df = pd.read_csv(file)
    # Load the pre-trained model
    path_pretrain_model = "../Result/MSF_DTI_BioSNAP_model.pt"

    model = DrugTargetModel().to(device)
    ckpt = torch.load(path_pretrain_model, map_location=device)
    model.load_state_dict(ckpt)
    model.eval()

    all_results = []
    with torch.no_grad():
        for batch in test_iter:
            X_hot, smile_mask, graph, kbert, protein_seq, protein_mask, esm, labels = batch
            graph = graph.to(device)
            kbert = kbert.to(device)
            protein_seq = protein_seq.to(device)
            esm = esm.to(device)
            logits = model(graph, kbert, protein_seq, esm)
            prob = torch.softmax(logits, dim=1)
            prob_positive = prob.cpu().numpy()[:, 1]
            pred_label = torch.argmax(logits, dim=1).cpu().numpy()

            for prob_val, true_label, pred in zip(
                    prob_positive,
                    labels.cpu().numpy(),
                    pred_label):
                all_results.append({
                    'Probability': prob_val,
                    'True_Label': true_label,
                    'Pred_Label': pred
                })

    results_df = pd.DataFrame(all_results)
    print("\nPredicted Results：")
    print(results_df)
    results_df.to_csv(
        'predict.csv',
        index=False,
        encoding='utf-8-sig'
    )


Predicted Results：
   Probability  True_Label  Pred_Label
0     0.000149           0           0
1     0.000159           0           0
2     0.999582           1           1
3     0.999491           1           1
4     0.000150           0           0
5     0.999636           1           1
6     0.999148           1           1
7     0.000149           0           0
8     0.982961           1           1
